# Day 25 · Simulated customers: tests that talk back

**First-principles question:** What breaks when the customer replies, switches language or changes their mind?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import numpy as np
from lab.agent import SYSTEM_C
from lab.evals import score
from lab.llm import chat
import lab.acme_tools as acme_tools

## 1 · Personas with a hidden goal and a twist

In [ ]:
PERSONAS = [
    {"id": "p1", "customer_id": "AC1001", "goal": "Find out about the outage and get the ticket id",
     "twist": "Switches to Hinglish on turn 2", "expected_intent": "outage", "needs_human": False,
     "must_include": ["OUT-KOTH-1021"], "must_not": []},
    # TODO: five more: a goal change, a wrong customer id first, anger, a refund above ₹500, a question about another customer
]

## 2 · Simulate a conversation

In [ ]:
def agent_turn(messages):
    """One turn of version C: runs tools until the model answers in text."""
    for _ in range(6):
        r = chat(messages, tools=acme_tools.TOOLS)
        messages.append(r["message"])
        if not r["tool_calls"]:
            return r["content"]
        for call in r["tool_calls"]:
            name, args = call["function"]["name"], call["function"]["arguments"]
            result = acme_tools.FUNCTIONS[name](**args) if name in acme_tools.FUNCTIONS else {"error": "unknown tool"}
            messages.append({"role": "tool", "tool_name": name, "content": json.dumps(result, ensure_ascii=False)})
    return ""

def simulate(persona, run=0, max_turns=4):
    """Returns the transcript: a list of (customer, agent) pairs."""
    # TODO: a second chat() call plays the customer: system prompt = the persona's goal and twist, it sees the
    #       conversation so far and writes the next customer message, or "DONE" when the goal is reached.
    #       Each turn: customer message → agent_turn(...) → append (customer, agent). Stop at DONE or max_turns.
    ...

## 3 · pass^1 and pass^3

In [ ]:
def grade(persona, transcript):
    """A conversation passes when every agent turn passes lab.evals.score (reply-level checks only)."""
    case = {k: persona[k] for k in ("expected_intent", "needs_human", "must_include", "must_not")}
    # TODO: score each agent reply as {"intent": persona["expected_intent"], "reply": reply, "needs_human": ...}
    #       and return True only when policy_ok holds on every turn and must_include holds by the last turn
    ...

results = {p["id"]: [grade(p, simulate(p, run=k)) for k in range(3)] for p in PERSONAS}
for pid, r in results.items():
    print(f"{pid}: pass^1 {np.mean(r):.0%}   pass^3 {all(r)}")

**The worst transcript, and what went wrong:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(PERSONAS) >= 6, "Step 1: six personas"
assert all(p["twist"] for p in PERSONAS)
assert all(len(r) == 3 for r in results.values()), "Step 3: three runs per persona"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")